In [43]:
#Starter-script for Prophet-konkurranse
#--------------------------------------
#
#- Leser train.csv
#- Leser test_features.csv
#- Trener Prophet-modell
#- Lager submission.csv

#Du kan forbedre:
#- features
#- skalering
#- lag-features
#- hyperparametre
#- log-transform
#- osv.

In [44]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.preprocessing import StandardScaler
import holidays

In [45]:
lagnsavn="testing_holi"

In [53]:
# --------------------------------------------------
# 1. LES DATA
# --------------------------------------------------
print("Leser data ...")
train = pd.read_csv("https://raw.githubusercontent.com/jensmorten/sykkelprofet/refs/heads/main/konkurranse/bysykkel_train.csv", parse_dates=["ds"])
test = pd.read_csv("https://raw.githubusercontent.com/jensmorten/sykkelprofet/refs/heads/main/konkurranse/test_compete.csv", parse_dates=["ds"])
test_truth=pd.read_csv("test_target_secret.csv", parse_dates=["ds"])
train = train.sort_values("ds")
test = test.sort_values("ds")
test_truth=test_truth.sort_values("ds")
print("done!")

Leser data ...
done!


In [47]:
weather_cols = [
    "air_temperature",
    "wind_speed",
    "precipitation_amount"
]

train[weather_cols] = (
    train[weather_cols]
    .ffill()
    .bfill()
)

In [48]:
regressors = [
    "air_temperature",
    "wind_speed",
    "precipitation_amount",
    "month",
    "weekofyear"
]

In [49]:
# --------------------------------------------------
# 4. DEFINER PROPHET
# --------------------------------------------------

m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=True,
    seasonality_mode="multiplicative",
)

m.add_country_holidays(country_name='NO')

#for r in regressors:
#    m.add_regressor(r)

In [50]:
print("Trener modell ...")
#m.fit(train[["ds", "y"] + regressors])
m.fit(train[["ds", "y"]])

Trener modell ...


12:05:48 - cmdstanpy - INFO - Chain [1] start processing
12:06:11 - cmdstanpy - INFO - Chain [1] done processing


In [51]:
# --------------------------------------------------
# 6. PREDIKSJON PÅ TEST
# --------------------------------------------------
future = test[["ds"] + regressors].copy()
future = test[["ds"]].copy()
forecast = m.predict(future)


In [52]:
submission = pd.DataFrame({
    "ds": test["ds"],
    "yhat": forecast["yhat"]
})

# Unngå negative prediksjoner
submission["yhat"] = submission["yhat"].clip(lower=0)

submission.to_csv(f"submission_{lagnsavn}.csv", index=False)

print("✅ Ferdig!")
print("Submission lagret")

✅ Ferdig!
Submission lagret
